# Numerical Analysis

Extracts **statistics, correlations, and binned peak/valley trends** for every
scalar parameter in the text data, plus prompt terms, discrete categories,
face/pose/hand struct fields, and positional data.

Output is a single nested JSON-like dict built entirely from Python built-in
types so it can be serialised or passed directly to an agent for review.

## 1. Imports and setup

In [ ]:
import sys, os, json, math
from pathlib import Path
from collections import OrderedDict
from typing import Any

import numpy as np
from tqdm import tqdm
from scipy.stats import pearsonr, spearmanr
from statistics import mean, stdev

if __name__ == "__main__":
    module_root = str(Path(os.getcwd()).parents[3])
    sys.path.insert(0, module_root)
    current_dir = str(Path(os.getcwd()).parents[2])
    sys.path.insert(0, current_dir)
    if __package__ is None:
        __package__ = (
            "comfyui_image_scorer.external_modules.training_hyperparameters.text_data"
        )

from ....shared.paths import text_data_file, scores_file
from ....shared.io import load_single_jsonl
from ....shared.config import config
from ....shared.io import write_single_jsonl
from ....shared.logger import configure_package_logging, get_logger, ModuleLogger
from .numerical_analysis import (
    extract_lora,
    extract_prompts,
    extract_face_logits,
    extract_face_bbox,
    extract_pose_landmarks,
    extract_hand_landmarks,
    extract_image_sizes,
    extract_remaining_fields,
    split_discrete,
    split_continuous,
    _group_continuous_raw,
    _group_str_discrete_raw,
    _group_int_discrete_raw,
    _group_positional_raw,
    _group_prompt_term_raw,
    _safe_stats,
    bucket_continuous,
    param_to_buckets,
    discrete_to_buckets,
    prompt_to_buckets,
    positional_to_buckets,
    flatten_results,
    flatten_to_jsonl_entries,
)

logger: ModuleLogger = get_logger(__name__)
configure_package_logging(level=10)

print(f"text_data_file: {text_data_file}")
print(f"scores_file:    {scores_file}")


## 2. Load data

In [ ]:
text_data = list(load_single_jsonl(text_data_file))
scores = list(load_single_jsonl(scores_file))
print(f"text data: {len(text_data)} lines")
print(f"scores:    {len(scores)} lines")
assert len(text_data) == len(scores), f"Mismatch: {len(text_data)} vs {len(scores)}"


## 3. Aggregate data using extractors from numerical_analysis module

In [ ]:
# Types imported from numerical_analysis module
from .numerical_analysis import (
    ElementType,
    ElementList,
    FloatKeyDict,
    IntKeyDict,
    StrKeyDict,
    PromptType,
    ContinuousType,
    PositionalType,
    FlattenedType,
)

# TODO: make sure everything below follows this exact structure
# then, flatten all elements following the flattened type format
# finally, make statistics

# the intermediate result should be the raw list with the new structure
# final result should be flattened ungrouped list with statistics results
# for final result consider using jsonl existing functions in io file.

# all functions should just be called but not defined in any part in this notebook
# ENDTODO

lora_data: ContinuousType = {}
positive_prompt: PromptType = {}
negative_prompt: PromptType = {}
discrete_data_str: StrKeyDict = {}
discrete_data_int: IntKeyDict = {}
continuous_data: ContinuousType = {}
face_age_data: ContinuousType = {}
face_gender_data: ContinuousType = {}
face_race_data: ContinuousType = {}
pose_data: ContinuousType = {}
lh_data: ContinuousType = {}
rh_data: ContinuousType = {}
positional_data: PositionalType = {}

PROCESS_LIMIT: int | None = None  # set to None for all records
MIN_BUCKET_COUNT: int = 1  # minimum samples required for a bucket to be included
total_records: int = (
    min(len(text_data), PROCESS_LIMIT) if PROCESS_LIMIT else len(text_data)
)

with tqdm(total=total_records, desc="Processing", unit="line") as pbar:
    for i in range(total_records):
        outer = text_data[i]
        current_score: float = float(scores[i])
        current_line: dict[str, Any] = dict(outer[next(iter(outer))])

        # prompt_texts.append(current_line.get("positive_prompt", ""))

        extract_lora(current_line, current_score, i, lora_data)
        extract_prompts(
            current_line, current_score, i, positive_prompt, negative_prompt
        )
        extract_face_logits(
            current_line,
            current_score,
            i,
            face_age_data,
            face_gender_data,
            face_race_data,
        )
        extract_face_bbox(
            current_line, current_score, i, continuous_data, positional_data
        )
        extract_pose_landmarks(
            current_line, current_score, i, pose_data, positional_data
        )
        extract_hand_landmarks(
            current_line, current_score, i, lh_data, rh_data, positional_data
        )
        extract_image_sizes(
            current_line, current_score, i, continuous_data, positional_data
        )
        extract_remaining_fields(
            current_line,
            current_score,
            i,
            discrete_data_str,
            discrete_data_int,
            continuous_data,
        )

        pbar.update(1)

# Keep the raw, pre-bucketing values for an intermediate export
raw_values = {
    "continuous": _group_continuous_raw(continuous_data),
    "lora": _group_continuous_raw(lora_data),
    "discrete_str": _group_str_discrete_raw(discrete_data_str),
    "discrete_int": _group_int_discrete_raw(discrete_data_int),
    "face_age": _group_continuous_raw(face_age_data),
    "face_gender": _group_continuous_raw(face_gender_data),
    "face_race": _group_continuous_raw(face_race_data),
    "pose": _group_continuous_raw(pose_data),
    "left_hand": _group_continuous_raw(lh_data),
    "right_hand": _group_continuous_raw(rh_data),
    "positional": _group_positional_raw(positional_data),
    "prompt_terms": {
        "positive": _group_prompt_term_raw(positive_prompt),
        "negative": _group_prompt_term_raw(negative_prompt),
    },
}

# ── Convert to grouped-value structure for bucketing ──

continuous_bounded, continuous_unbounded = split_continuous(continuous_data)

# Split discrete — already in correct format
discrete_numeric: IntKeyDict = discrete_data_int
discrete_labels: StrKeyDict = discrete_data_str

print(f"\ncontinuous fields: {len(continuous_bounded) + len(continuous_unbounded)}")
print(f"  bounded (0-1):   {list(continuous_bounded.keys())}")
print(f"  unbounded:       {list(continuous_unbounded.keys())}")
print(f"discrete fields:   {len(discrete_numeric) + len(discrete_labels)}")
print(f"  numeric:         {list(discrete_numeric.keys())}")
print(f"  label:           {list(discrete_labels.keys())}")
print(f"lora entries:      {len(lora_data)}")
print(f"positive terms:    {len(positive_prompt)}")
print(f"negative terms:    {len(negative_prompt)}")


## 4. Helper functions for numerical analysis

In [ ]:
# All helpers are imported from numerical_analysis module
# MIN_BUCKET_COUNT and all bucket functions are defined there


## 5. Run analysis on all parameters

In [ ]:
all_results: dict[str, dict] = {}

print("=== Continuous (bounded + unbounded) ===")
all_c = dict(continuous_bounded)
all_c.update(continuous_unbounded)
all_results["continuous"] = param_to_buckets(all_c)
print(f"  {len(all_results['continuous'])} parameters")

print("=== Lora ===")
all_results["lora"] = param_to_buckets(lora_data, prefix="weight_")
print(f"  {len(all_results['lora'])} loras")

print("=== Discrete numeric ===")
all_results["discrete_numeric"] = discrete_to_buckets(discrete_numeric)
print(f"  {len(all_results['discrete_numeric'])} parameters")

print("=== Discrete labels ===")
all_results["discrete_labels"] = discrete_to_buckets(discrete_labels)
print(f"  {len(all_results['discrete_labels'])} parameters")

print("=== Face attributes ===")
all_results["face_age"] = param_to_buckets(face_age_data)
all_results["face_gender"] = param_to_buckets(face_gender_data)
all_results["face_race"] = param_to_buckets(face_race_data)
print(f"  age/ gender/ race done")

print("=== Pose landmarks ===")
all_results["pose"] = param_to_buckets(pose_data)
print(f"  {len(all_results['pose'])} fields")

print("=== Hand landmarks ===")
all_results["left_hand"] = param_to_buckets(lh_data)
all_results["right_hand"] = param_to_buckets(rh_data)
print(f"  left/right done")

print("=== Positional ===")
all_results["positional"] = positional_to_buckets(positional_data)
print(f"  {len(all_results['positional'])} fields")

print("=== Prompt terms ===")
all_results["prompt_terms"] = {
    "positive": prompt_to_buckets(positive_prompt),
    "negative": prompt_to_buckets(negative_prompt),
}
print(f"  positive: {len(all_results['prompt_terms']['positive'])} terms")
print(f"  negative: {len(all_results['prompt_terms']['negative'])} terms")

print("\nDone — All analyses complete")


## 6. Export to JSON for agent consumption

In [ ]:
from ....shared.paths import output_dir

intermediate_path = Path(output_dir) / "numerical_analysis_raw_values.json"

with open(intermediate_path, "w", encoding="utf-8") as f:
    json.dump(raw_values, f, indent=2, ensure_ascii=False, default=str)

flat = flatten_results(all_results)
entries = flatten_to_jsonl_entries(flat)
final_path = Path(output_dir) / "numerical_analysis_results.jsonl"
write_single_jsonl(str(final_path), entries, "w")

print(f"Exported flattened results to {final_path}")
print(f"  {len(entries)} entries")
print(f"Intermediate raw values to {intermediate_path}")
print(f"Intermediate size: {intermediate_path.stat().st_size / 1024:.1f} KB")


## 7. Quick-look summary of notable findings

In [ ]:
def _show_sample(d, depth=0):
    if isinstance(d, dict) and "count" in d and "mean_score" in d:
        print("  " * depth + "Leaf:", json.dumps(d, indent=2))
        return True
    if isinstance(d, dict):
        for k, v in d.items():
            print("  " * depth + k, end="")
            if isinstance(v, dict):
                if "count" in v:
                    print(" leaf")
                    print(
                        "  " * (depth + 1)
                        + json.dumps(v, indent=2).replace(
                            "\n", "\n" + "  " * (depth + 1)
                        )
                    )
                    return True
                else:
                    print()
                    if _show_sample(v, depth + 1):
                        return True
            else:
                print(" =", v)
    return False


print("=== Structure sample: prompt_terms.positive ===")
pos = all_results.get("prompt_terms", {}).get("positive", {})
if pos:
    first = list(pos.keys())[0]
    print(f'"{first}" -> ')
    _show_sample(pos[first])


## 8. Config-based parameter reference (from vector_config.json)

In [ ]:
# Show which fields from vector_config.json we analyzed and which we skipped
vec_config = config["vector"]["vectors"]
print(f"{'Name':30s} {'Type':15s} {'Status':20s}")
print("-" * 65)
for v in vec_config:
    name = v["name"]
    vtype = v["type"]
    if vtype in ("image", "embedding"):
        status = "excluded"
    elif name in all_results.get("continuous", {}):
        status = "continuous"
    elif name in all_results.get("discrete_numeric", {}):
        status = "discrete_numeric"
    elif name in all_results.get("discrete_labels", {}):
        status = "discrete_label"
    elif name in all_results.get("positional", {}):
        status = "positional"
    elif name == "face_logits":
        status = "face_attrs (age/gender/race)"
    elif name == "face_bbox":
        status = "face_bbox (in positional)"
    elif name == "body_pose":
        status = "pose_landmarks"
    elif name in ("left_hand", "right_hand"):
        status = "hand_landmarks"
    elif name == "nsfw_score":
        status = "continuous"
    else:
        status = "not_found_in_analysis"
    print(f"{name:30s} {vtype:15s} {status:20s}")
